# Analyse Helios Simulations
This notebook walks through the process of analysing helios simulations which follows 3 main steps:

    (1) Data Preparation
        This stage will setup the project directory, setup expected schemas for dataframes (both dask and pandas), and ultimately read in the helios data and prepare the required per ray information into a .parquet output.
        It will also setup the reference dataset for voxels for each voxel_size in the project (i.e. unique voxel_ids etc.).
    
    (2) Voxel-Ray Intersection
        With valid rays saved per leg of the scan, in the previous step, the goal now is to check ray intersections in all voxels. This will record important information, such as the entry/exit/hit coordinates of the ray which will later be used to gather metrics.
        The main reason these metrics are not gathered yet, is that this stage will remain separate per leg. That way, the metrics can be computed from different combinations of helios legs without re-computing voxel-ray intersections.

    (3) Compute Metrics
        Taking a given set of legs and voxel_sizes, the voxel_ray intersection files will be used to calculate metrics for each voxel, in this case resulting in all outputs from each investigated method.

# Step 1 - Setup Project
Set project paths here

In [ ]:
import os

# Set up the project directory
project_dir = '/home/capheus/projects/1001_etri_erecto_diamond'
helios_dir = os.path.join(project_dir, 'helios')
references_dir = os.path.join(project_dir, 'references')
results_dir = os.path.join(project_dir, 'results')
valid_rays_dir = os.path.join(project_dir, 'valid_rays')

# Establish output path for the voxel_ray_intersections parquet folder
voxel_ray_intersections_parquet = os.path.join(valid_rays_dir, 'voxel_ray_intersections')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir, exist_ok=True)

if not os.path.exists(results_dir):
    os.makedirs(results_dir, exist_ok=True)

use_class = False
leaf_object_ids = [0]
wood_object_ids = [1]

### Debug classification settings

In [ ]:
import csv
import glob
import numpy as np
import matplotlib.pyplot as plt

# Check classification and object_ids
helios_files = glob.glob(os.path.join(helios_dir, '*.xyz'))
if helios_files:
    test_file = helios_files[0]     # Just use the first file
    class_col = 9 if use_class else 8      # if not use_class, assume use hit_object_id
    num_test_points = 1000

    num_rows = 0
    leaf_points = []
    wood_points = []
    other_points = []
    with open(test_file, newline="") as f:
        reader = csv.reader(f, delimiter=' ')
        while num_rows < num_test_points:
            for row in reader:
                x = float(row[0])
                y = float(row[1])
                z = float(row[2])

                class_id = int(row[class_col])
                if class_id in leaf_object_ids:
                    leaf_points.append([x,y,z])
                elif class_id in wood_object_ids:
                    wood_points.append([x,y,z])
                else:
                    other_points.append([x,y,z])
                num_rows += 1

    if len(leaf_points) > 0 or len(wood_points) > 0 or len(other_points) > 0:
        # Convert to numpy
        leaf_points = np.array(leaf_points, dtype=np.float32)
        wood_points = np.array(wood_points, dtype=np.float32)
        other_points = np.array(other_points, dtype=np.float32)
        
        # Plot point cloud
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(111)

        # Plot leaf points in green
        if leaf_points.size > 0:
            ax.scatter(leaf_points[:, 0], leaf_points[:, 2], c='green', s=1, label='Leaf')

        # Plot wood points in brown
        if wood_points.size > 0:
            ax.scatter(wood_points[:, 0], wood_points[:, 2], c='saddlebrown', s=1, label='Wood')

        # Plot other points in blue
        if other_points.size > 0:
            ax.scatter(other_points[:, 0], other_points[:, 2], c='blue', s=1, label='Other')

        print("Plotting leaf and wood points to check classification...")
        ax.set_xlabel('X')
        ax.set_ylabel('Z')
        ax.set_title(f'Leaf and Wood Point Check - File {os.path.basename(test_file)}')
        ax.legend()
        plt.show()
        plt.savefig(os.path.join(valid_rays_dir, f'file_{os.path.basename(test_file)}_leaf_wood_check.png'))

## Step 1 - Data Preparation
This step focuses on converting helios simulation outputs, saving only valid rays into a more efficient .parquet file format.

It expects the following input and will add a new folder (valid_rays) to store all resulting .parquet files.

INPUT:
    project_dir/
    ├── reference/
    │   ├── "{project}_voxel_size_0.2.csv"
    │   ├── "{project}_voxel_size_0.5.csv"
    │   ...
    │   └── "{project}_voxel_size_{v}.csv"
    ├── helios/
    │   ├── "leg000_points.xyz"
    │   ├── "leg000_pulse.txt"
    │   ├── "leg000_fullwave.txt"
    │   ├── "leg001_points.xyz"
    │   ├── "leg001_pulse.txt"
    │   ├── "leg001_fullwave.txt"
    │   ├── ...
    │   ├── "leg{l}_points.xyz"
    │   ├── "leg{l}_pulse.txt"
    │   └── "leg{l}_fullwave.txt"

OUTPUT:
    └── valid_rays/
        ├── "leg_000_valid_rays.parquet"
        ├── "leg_001_valid_rays.parquet"
        ...
        └── "leg_{l}_valid_rays.parquet"

In [ ]:
from utils import prepare_helios_data

# Run the data preparation script
prepare_helios_data(
    input_dir=helios_dir, 
    output_dir=valid_rays_dir, 
    references_dir=references_dir, 
    leaf_object_ids=leaf_object_ids, 
    wood_object_ids=wood_object_ids, 
    use_class=use_class
)

### Step 1.5 -  Compute Normals and Weights for Leaf Points

In [ ]:
from utils import add_normals_weights_to_valid_rays

# Calculate normals and weights by loading valid rays
add_normals_weights_to_valid_rays(
    valid_rays_dir, 
    debug=True,
    knn=6
)

## Step 2 - Voxel Ray Intersections
This code uses the valid rays from before, alongside the reference datasets in order to create a supporting parquet in the valid rays folder using the voxel_size_{voxel_size}_leg_{leg}_intersections.parquet format.

In [ ]:
from utils import voxel_ray_intersections_dask_new

# Run intersections
voxel_ray_intersections_dask_new(
    valid_rays_dir=valid_rays_dir, 
    references_dir=references_dir,
    output_path=voxel_ray_intersections_parquet,
    optimal_threads=8,
    debug=False
)

## Step 3 - Compute Metrics
Using the leg_{leg_id}_voxel_size_{voxel_size}_intersections.parquet files (which feature a standardised structure of columns from various inputs), compute the desired metrics and save outputs.

In [9]:
import os
import glob
import utils
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import dask.dataframe as dd
from utils import calculate_lambda_1, get_voxel_metrics

# Select the desired legs and voxel_sizes to include in the analysis
# Use the shortcut string 'all' to include all 
legs = [0, 1, 2, 3] # 'all' # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
voxel_sizes = [0.2, 0.5, 1.0, 2.0]
# Set the average leaf area
average_leaf_area = 0.003582  # in m^2, adjust as needed

leg_string = None if not legs == 'all' else legs

# Estabilish voxel_ray_intersections parquet folder structure
if not os.path.exists(voxel_ray_intersections_parquet):
    raise FileNotFoundError("The voxel_ray_intersections parquet directory does not exist. Please check the path.")

# # Filter parquet to create dask dataframe of chosen leg and voxel_size combinations
# part_schema = pa.schema([
#     pa.field("leg_id", pa.int32()),
#     pa.field("voxel_size", pa.float32())
# ])

# dataset = ds.dataset(
#     voxel_ray_intersections_parquet,
#     format="parquet",
#     partitioning=ds.HivePartitioning(part_schema),
# )

# if dataset is None:
#     raise ValueError("The dataset could not be loaded. Please check the parquet directory.")



for voxel_size in voxel_sizes:
    
    # if legs == 'all':
    #     filt = (ds.field("voxel_size") == pa.scalar(float(voxel_size), pa.float32()))
    # else:
    #     filt = (
    #         (ds.field("leg_id").isin([int(l) for l in legs])) &
    #         (ds.field("voxel_size") == pa.scalar(float(voxel_size), pa.float32()))
    #     )

    # Calculate lambda_1
    lambda_1 = calculate_lambda_1(voxel_size=voxel_size, average_leaf_area=average_leaf_area)

    if legs == 'all':
        filt = [('voxel_size', '==', str(voxel_size))]
    else:
        filt = [
            ('leg_id', 'in', [str(l) for l in legs]),
            ('voxel_size', '==', str(voxel_size))
        ]

    # Establish dask dataframe from parquet
    voxel_ray_intersections_ddf = dd.read_parquet(
        voxel_ray_intersections_parquet,
        engine='pyarrow',
        filters=filt
    )

    ### DEBUG
    # Save voxel_ray_intersections_ddf to csv for inspection
    voxel_ray_intersections_ddf.to_csv(
        os.path.join(valid_rays_dir, f"voxel_ray_intersections_{leg_string}_voxel_size_{voxel_size}.csv"),
        index=False,
        single_file=True
    )
    print(f"Saved voxel_ray_intersections_ddf for voxel_size {voxel_size} and legs {leg_string} to CSV for debugging.")

    # results_df = get_voxel_metrics(
    #     intersections_ddf=voxel_ray_intersections_ddf,
    #     lambda_1=lambda_1,
    #     is_multireturn=False
    # )

    # results_df.to_csv(
    #     os.path.join(results_dir, f"voxel_metrics_{leg_string}_voxel_size_{voxel_size}.csv"),
    #     index=False
    # )

Saved voxel_ray_intersections_ddf for voxel_size 0.2 and legs None to CSV for debugging.
Saved voxel_ray_intersections_ddf for voxel_size 0.5 and legs None to CSV for debugging.
Saved voxel_ray_intersections_ddf for voxel_size 1.0 and legs None to CSV for debugging.
Saved voxel_ray_intersections_ddf for voxel_size 2.0 and legs None to CSV for debugging.


# Extra Tools

In [3]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.csv as pacsv

# Set parquet folder path
parquet_folder = os.path.join(valid_rays_dir, "voxel_ray_intersections")

# Set desired legs and voxel sizes
voxel_sizes = [1.0]  # [0.2, 0.5, 1.0, 2.0]
legs = [0] # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

part_schema = pa.schema([
    pa.field("leg_id", pa.int32()),
    pa.field("voxel_size", pa.float32())
])

dataset = ds.dataset(
    parquet_folder,
    format="parquet",
    partitioning=ds.HivePartitioning(part_schema),
)

for leg in legs:
    for voxel_size in voxel_sizes:
        
        filt = (
            (ds.field("leg_id") == pa.scalar(int(leg), pa.int32())) &
            (ds.field("voxel_size") == pa.scalar(float(voxel_size), pa.float32()))
        )

        table = dataset.to_table(filter=filt)

        out_csv = os.path.join(
            valid_rays_dir, 
            f"voxel_ray_intersections_leg_{leg}_voxel_size_{voxel_size}.csv"
        )
        
        pacsv.write_csv(table, out_csv)
        print(f"Exported leg {leg}, voxel size {voxel_size} to CSV.")

Exported leg 0, voxel size 1.0 to CSV.


OLD VOXEL_RAY_INTERSECTIONS TEST

In [8]:
import glob
import dask.dataframe as dd

intersection_files = glob.glob(os.path.join(valid_rays_dir, "*2.0_intersections.csv"))

ddfs = []
for file in intersection_files:
    ddf = dd.read_csv(file)
    ddfs.append(ddf)

intersections_ddf = dd.concat(ddfs)

intersections_ddf.to_csv(
    os.path.join(valid_rays_dir, f"voxel_ray_intersections_all_legs_voxel_size_2.0_OLD.csv"),
    index=False,
    single_file=True
)

['/home/capheus/projects/1001_etri_erecto_diamond/valid_rays/voxel_ray_intersections_all_legs_voxel_size_2.0_OLD.csv']